# Trading Intelligent — CH02 v4.0
## Laboratoire 3 — Saisonnalité, différenciation saisonnière, SARIMA et Fourier

**Auteur académique : Tawfik Masrour**

**Position dans le cours :** Chapitre 2 — *Comprendre la dépendance temporelle : stationnarité, mémoire, ARIMA et saisonnalité*.

### Question directrice
Comment reconnaître une répétition périodique, comprendre pourquoi la différence ordinaire ne suffit pas, puis choisir entre structure saisonnière stochastique et représentation harmonique ?

> **Contrat pédagogique du laboratoire.** On ne demande jamais au code de nous dire ce que nous n'avons pas encore défini. Chaque expérience suit le fil : **question → intuition → calcul → graphique → diagnostic → interprétation financière**.

> Ce notebook reprend intégralement les expériences pédagogiques du laboratoire historique correspondant, mais les replace dans le cours **Trading Intelligent** et les enrichit par des explications, contrôles et ponts vers la finance.


### 0. La saisonnalité avant SARIMA

Une saisonnalité est une structure qui se répète selon une **période \(s\)**.

Exemples :
- données mensuelles avec cycle annuel : \(s=12\) ;
- données trimestrielles avec cycle annuel : \(s=4\) ;
- données horaires avec motif journalier : \(s=24\), si le calendrier est réellement régulier.

Une forme déterministe simple est

\[
S_t=A\sin\left(
rac{2\pi t}{s}+
arphi
ight),
\]

où \(A\) est l'amplitude et \(
arphi\) la phase.

Mais une saisonnalité peut aussi être **stochastique** : \(X_t\) peut dépendre directement de \(X_{t-s}\). C'est cette seconde idée qui conduit naturellement aux opérateurs saisonniers et à SARIMA.


### Calcul manuel — différence ordinaire versus différence saisonnière

Supposons des ventes mensuelles :

| Mois | Jan. 2025 | Fév. 2025 | ... | Jan. 2026 |
|---|---:|---:|---:|---:|
| Valeur | 100 | 108 | ... | 125 |

La différence ordinaire de février vaut

\[
\Delta X_{	ext{fév. 2025}}=108-100=8.
\]

La différence saisonnière de janvier 2026 vaut

\[
\Delta_{12}X_{	ext{jan. 2026}}
=125-100=25.
\]

Les deux opérations ne répondent pas à la même question :

- \(\Delta X_t\) compare à la période immédiatement précédente ;
- \(\Delta_{12}X_t\) compare à la **même position du cycle précédent**.

Avec l'opérateur retard \(L\) :

\[
\Delta X_t=(1-L)X_t,\qquad
\Delta_sX_t=(1-L^s)X_t.
\]


##**1️⃣ Setup & Installation**
Le modèle SARIMA est complexe (7 paramètres). Pour trouver les meilleurs paramètres automatiquement, nous utiliserons la librairie standard de ***pmdarima***.

In [ ]:
# Installation des outils nécessaires
!pip install pmdarima

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import acf, pacf, adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import seasonal_decompose

# Librairie Auto-ARIMA
import pmdarima as pm

# Config Esthétique
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print("✅ Environnement SARIMA chargé.")

#**2️⃣ Partie I : Simulation "À la Main"**

Nous allons fabriquer une série temporelle artificielle qui contient trois ingrédients connus :

**Une Tendance ($d=1$)** : Elle monte régulièrement.

**Une Saisonnalité ($s=12$)** : Un motif qui se répète tous les 12 mois.

**Du Bruit** : Pour rendre le tout réaliste.

**Équation Mathématiqu**e :$$Y_t = \underbrace{10 + 0.05t}_{\text{Tendance}} + \underbrace{2 \sin\left(\frac{2\pi t}{12}\right)}_{\text{Saisonnalité}} + \underbrace{\epsilon_t}_{\text{Bruit}}$$

In [ ]:
# --- GÉNÉRATION DE LA SÉRIE SAISONNIÈRE ---
np.random.seed(42)
n_obs = 144 # 12 années de 12 mois
t = np.arange(n_obs)

# 1. Composante Tendance (Linéaire)
trend = 0.05 * t + 10

# 2. Composante Saisonnière (Sinusoïdale pure, période 12)
seasonality = 2 * np.sin(2 * np.pi * t / 12)

# 3. Bruit Blanc
noise = np.random.normal(0, 0.5, n_obs)

# 4. Série Totale
y_sim = trend + seasonality + noise

# --- VISUALISATION ---
plt.figure(figsize=(14, 6))
plt.plot(y_sim, label='Série Simulée (Tendance + Saison)', color='#3b82f6')
plt.title("Notre création : Une série avec une forte saisonnalité annuelle ($s=12$)")
plt.legend()
plt.show()

# Décomposition Rapide pour preuve
res = seasonal_decompose(y_sim, period=12)
res.plot()
plt.show()

#**3️⃣ Le Piège de la Différenciation Simple ($d=1$)**

Beaucoup pensent que diff() suffit à tout régler. C'est faux.

Regardons ce qui se passe si on applique seulement une différence simple ($y_t - y_{t-1}$).

### Pourquoi \(d=1\) peut échouer face à une saisonnalité

Une différence première annule efficacement une composante linéaire parce que

\[
(a+bt)-(a+b(t-1))=b.
\]

Mais si la série possède une structure qui revient tous les 12 mois, soustraire le mois précédent ne compare pas deux positions saisonnières équivalentes. Le motif peut donc rester très visible.

C'est précisément ce que l'expérience suivante doit montrer : **on constate d'abord l'échec, puis on introduit l'outil qui répond à cet échec**.


In [ ]:
# --- TENTATIVE 1 : Différenciation Simple (d=1) ---
y_diff_1 = np.diff(y_sim, n=1)

# Visualisation
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Série Différenciée
ax[0].plot(y_diff_1, color='#f59e0b')
ax[0].set_title("Après Différenciation Simple ($d=1$)\nLa tendance est partie, MAIS...")

# ACF (Le Juge de Paix)
plot_acf(y_diff_1, lags=36, ax=ax[1], title="ACF : La Signature du problème")
plt.show()

##📝 **Analyse des Graphiques (Preuve par l'Image)**

**1. Le Piège ($d=1$) :**
Regardez le graphique orange ("Après Différenciation Simple").
* La série ne monte plus vers le ciel (Tendance tuée ✅).
* **MAIS** elle continue d'osciller très fort régulièrement.
* L'ACF montre des pics périodiques évidents. ***C'est la signature de la saisonnalité qui a survécu***.



##4️⃣ La Solution : La Différenciation Saisonnière ($D=1$)

Pour tuer la saison, il faut comparer ce mois-ci avec le même mois de l'année dernière.

$$\Delta_{12} Y_t = Y_t - Y_{t-12}$$

### La logique de \(D=1\)

La différence saisonnière

\[
(1-L^{12})X_t=X_t-X_{t-12}
\]

compare janvier à janvier, février à février, etc. Si une composante saisonnière identique se répète exactement,

\[
S_t=S_{t-12},
\]

alors

\[
S_t-S_{t-12}=0.
\]

Voilà pourquoi l'opérateur saisonnier peut faire disparaître une structure que la différence ordinaire laisse intacte.


In [ ]:
# --- TENTATIVE 2 : Différenciation Saisonnière (D=1, s=12) ---
# On soustrait à chaque point la valeur d'il y a 12 mois
y_diff_season = np.diff(y_sim, n=12)

# Visualisation
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Série
ax[0].plot(y_diff_season, color='#10b981')
ax[0].set_title("Après Différenciation Saisonnière ($D=1, s=12$)\nLa saisonnalité est cassée !")

# ACF
plot_acf(y_diff_season, lags=36, ax=ax[1], title="ACF : Plus de pics périodiques")
plt.show()

##📝**Analyse Expert (La Solution ($D=1$))**

Regardez le **graphique vert** ("Après Différenciation Saisonnière").

Ce qui a disparu (Le succès) :

Regardez les lags 12, 24 et 36 sur l'ACF : ils sont maintenant dans la zone bleue (proches de 0).

**Victoire** :
 .Ce qui

*   La mémoire "annuelle" a été effacée
*   Le cycle saisonnier est cassé
*   ***reste (Le travail pour ARIMA***) :L'ACF ne tombe pas immédiatement à plat.On observe un fort pic négatif au Lag 1 et quelques corrélations aux lags 2 et 3.
*   Interprétation : La série est devenue stationnaire (moyenne et variance stables), mais elle n'est pas encore un "Bruit Blanc". Il reste une dynamique à court terme que **les parties AR($p$) et MA($q$) du modèle devront expliquer**.
**Conclusion** : Le terrain est préparé (plus de saisonnalité), mais il reste du travail de modélisation fine à faire. C'est exactement pour cela que l'Auto-ARIMA va chercher des coefficients $p$ et $q$.

##**5️⃣ Partie II : Modélisation et Prévision de la Série Simulée**

Puisque nous avons réussi à stationnariser notre série artificielle (verte), demandons maintenant à un modèle SARIMA de l'apprendre et de prédire la suite.

Comme nous connaissons la vérité (c'est nous qui l'avons codée !), nous pourrons vérifier si le modèle est intelligent.

**Rappel de notre création** :Tendance linéaire ($d=1$).

**Saisonnalité de période 12 ($D=1, s=12$)**.

**Forme Sinusoïdale** : Attention, une courbe lisse (sinus) est difficile à reproduire pour un modèle linéaire simple. ***Il est probable que l'Auto-ARIMA ajoute des termes Auto-Régressifs (AR) pour réussir à "dessiner" la courbure des vagues***.

Voyons ce qu'il trouve :

In [ ]:
# --- 1. PRÉPARATION (Train / Test) ---
# On cache les 24 derniers mois (2 ans) pour tester la prévision
train_sim = y_sim[:-24]
test_sim = y_sim[-24:]

print(f"Entraînement sur {len(train_sim)} mois.")
print(f"Test sur {len(test_sim)} mois (futur caché).")

# --- 2. AUTO-ARIMA (Recherche du Modèle) ---
# Nous guidons un peu le modèle en lui disant que m=12 (mensuel)
print("⏳ Recherche du meilleur modèle...")

model_sim = pm.auto_arima(
    train_sim,
    seasonal=True, m=12,  # Indispensable : dire qu'il y a une saison !
    d=1, D=1,             # On sait qu'il y a une tendance et une saison
    start_p=0, start_q=0,
    max_p=2, max_q=2,     # Limite pour la partie court terme
    max_P=2, max_Q=2,     # AJOUT : Limite pour la partie saisonnière (Important car P=2 trouvé !)
    trace=True,
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True         # Algorithme intelligent
)

print("\n🏆 VERDICT DU MODÈLE :")
print(model_sim.summary())

##**📝 Analyse Détaillée du Verdict**

❗***Pourquoi ces paramètres ?***

**Les Différences (1, 1) **: Le modèle a validé qu'une différenciation simple ($d=1$) et une différenciation saisonnière ($D=1$) étaient nécessaires pour stabiliser la tendance et le cycle de 12 mois.

**L'ordre (2) et (2)** : C'est le point le plus subtil. **Notre série originale est une sinusoïde**. En mathématiques, une oscillation lisse ne peut pas être capturée par un simple décalage. Il faut au moins deux points de mémoire ($p=2$) pour que le modèle puisse calculer la "courbure" de la vague. L'algorithme a donc recréé notre sinus en utilisant deux poids sur le court terme (ar.L1, ar.L2) et deux poids sur le cycle annuel (ar.S.L12, ar.S.L24).

**Signification Statistique (La colonne $P>|z|$)**
Tous les coefficients ont une P-value de 0.000 ou 0.001.

**Verdict** : Ils sont tous hautement significatifs. Le modèle n'a pas ajouté de paramètres "au hasard" ; chaque coefficient aide réellement à réduire l'erreur.

**Diagnostic des Résidus (Les tests en bas)Ljung-Box ($Prob(Q) = 0.42$)** : On veut que ce chiffre soit supérieur à 0.05. Ici, 0.42 signifie qu'on ne peut pas rejeter l'hypothèse que les résidus sont un bruit blanc.

 Le modèle a capturé toute l'information !

 **Jarque-Bera ($Prob(JB) = 0.51$)** : Idem, supérieur à 0.05. Cela confirme que nos erreurs suivent une loi normale (pas de valeurs aberrantes monstrueuses).



In [ ]:
# --- 3. PRÉVISION (24 mois) ---
forecast_sim, conf_int_sim = model_sim.predict(n_periods=24, return_conf_int=True)

# --- 4. VISUALISATION ---
plt.figure(figsize=(14, 7))

# Historique récent
plt.plot(np.arange(len(train_sim)-36, len(train_sim)), train_sim[-36:], label="Historique (Train)", color='black')

# Réalité cachée (Test)
plt.plot(np.arange(len(train_sim), len(y_sim)), test_sim, label="Réalité (Vrai Sinus)", color='green', linestyle="--", linewidth=2)

# Prévision du modèle
plt.plot(np.arange(len(train_sim), len(y_sim)), forecast_sim, label="Prévision SARIMA", color='red', linewidth=2)

# Cône d'incertitude
plt.fill_between(
    np.arange(len(train_sim), len(y_sim)),
    conf_int_sim[:, 0],
    conf_int_sim[:, 1],
    color='pink', alpha=0.3, label="Intervalle 95%"
)

plt.title("Validation SARIMA : Reconstruction du motif sinusoïdal simulé")
plt.legend(loc='upper left')
plt.show()

##**📝 Conclusion de l'Expérience**

Observez la courbe rouge. Elle épouse parfaitement les vagues de la courbe verte.

**Victoire**: Le modèle a "compris" que le motif se répétait tous les 12 mois et l'a projeté dans le futur.

C'est la force du paramètre saisonnier ($P, D, Q)_s$.

##**1️⃣ Extraction des paramètres**
Ceci est un code pour récupérer les valeurs exactes de notre modèle approché model_sim (objet ARIMA de pmdarima) :

In [ ]:
# Extraction des paramètres proprement avec les noms
params = model_sim.arima_res_.params
names = model_sim.arima_res_.param_names

# Création d'un dictionnaire pour un accès facile
coef_dict = dict(zip(names, params))

# Extraction sécurisée
phi1 = coef_dict.get('ar.L1', 0)
phi2 = coef_dict.get('ar.L2', 0)
Phi1 = coef_dict.get('ar.S.L12', 0)
Phi2 = coef_dict.get('ar.S.L24', 0)
sigma2 = coef_dict.get('sigma2', 0)

print(f"--- Coefficients extraits ---")
print(f"Phi 1 (AR1)  : {phi1:.4f}")
print(f"Phi 2 (AR2)  : {phi2:.4f}")
print(f"SAR 1 (L12)  : {Phi1:.4f}")
print(f"SAR 2 (L24)  : {Phi2:.4f}")
print(f"Bruit (sigma2): {sigma2:.4f}")

##🔬 **Formalisation Mathématique de la Solution**

Le modèle sélectionné par l'algorithme est un SARIMA$(2,1,0) \times (2,1,0)_{12}$.

Ce résultat est fascinant car il montre comment un **modèle linéaire s'organise pour reconstruire une fonction non-linéaire (notre sinus)**.

**A.** **L'Équation aux Différences (Le Filtrage)**: Le modèle commence par transformer notre série brute (Tendance + Saison) en une série stationnaire $z_t$ via une double différenciation. C'est l'étape qui "aplatit" le terrain :$$z_t = \Delta \Delta_{12} y_t = (y_t - y_{t-1}) - (y_{t-12} - y_{t-13})$$

**B. L'Équation de Prédiction (La Mémoire)**: Pour la composante stationnaire $z_t$, le modèle a établi la règle de récurrence suivante en utilisant les coefficients extraits :$$\hat{z}_t = \underbrace{-0.6325 z_{t-1} - 0.3015 z_{t-2}}_{\text{Mémoire Court Terme (AR)}} \underbrace{- 0.6938 z_{t-12} - 0.3586 z_{t-24}}_{\text{Mémoire Saisonnière (SAR)}}$$

**C. Reconstruction de la Valeur Réelle ($y_t$)**: La prévision finale que nous voyons sur le graphique est obtenue en "dé-différenciant" le résultat. Le modèle projette la structure passée et lui ajoute la correction stochastique calculée ci-dessus :$$\hat{y}_t = \underbrace{y_{t-1} + (y_{t-12} - y_{t-13})}_{\text{Projection Déterministe}} + \underbrace{\hat{z}_t}_{\text{Ajustement des cycles}}$$

💡 **Interprétation d'Expert**
**L'Approximation du Sinus** : Notre série originale contenait un sinus pur. Le modèle ARIMA est intrinsèquement linéaire. Pour capturer la courbure fluide d'une sinusoïde, deux points de mémoire ($p=2$) sont indispensables : ils permettent au modèle d'estimer la "vitesse" et "l'accélération" de la courbe. C'est l'équivalent stochastique d'un développement de Taylor au second ordre.

**La Force de Rappel **: Les coefficients sont tous négatifs. C'est un signe de stabilité extrême. Si la série s'éloigne trop de sa trajectoire à cause du bruit ($\sigma^2 = 0.39$), ces coefficients négatifs agissent comme des ressorts qui ramènent immédiatement la prévision vers le cycle sinusoïdal théorique.

### Déchiffrer SARIMA sans mémoriser une recette

La notation

\[
SARIMA(p,d,q)	imes(P,D,Q)_s
\]

sépare deux échelles de mémoire :

- \((p,d,q)\) : dynamique **ordinaire** ;
- \((P,D,Q)\) : dynamique **saisonnière** ;
- \(s\) : longueur du cycle.

Une écriture compacte est

\[
\Phi(L)\Phi_s(L^s)(1-L)^d(1-L^s)^D X_t
=
\Theta(L)\Theta_s(L^s)
arepsilon_t.
\]

Chaque morceau a déjà un sens :
- \((1-L)^d\) retire les racines unitaires ordinaires ;
- \((1-L^s)^D\) retire les racines unitaires saisonnières ;
- les polynômes AR et MA ordinaires/saisonniers modélisent ce qui reste.


#1️⃣ **L'approche SARIMAX + Fourier**
Nous utilisons le FourierFeaturizer de pmdarima pour créer des régresseurs sinus/cosinus, puis nous les injectons dans un modèle ARIMA.

In [ ]:
from pmdarima.preprocessing import FourierFeaturizer
from pmdarima.arima import auto_arima

# 1. Création des régresseurs de Fourier (K=1 paire de sinus/cosinus)
# m=12 pour un cycle annuel
four_feat = FourierFeaturizer(m=12, k=1)

# .fit_transform renvoie une matrice (n, 2) contenant sin et cos
X_train_fourier, _ = four_feat.fit_transform(train_sim)
X_test_fourier, _ = four_feat.transform(test_sim)

# --- CORRECTION DIMENSION ---
# S'assurer que les exogènes sont bien traitées comme des matrices 2D
X_train_fourier = np.array(X_train_fourier).reshape(len(train_sim), -1)
X_test_fourier = np.array(X_test_fourier).reshape(len(test_sim), -1)

# 2. Entraînement du modèle "Trigonométrique"
# On règle seasonal=False car la saisonnalité est portée par X
model_fourier = auto_arima(train_sim,
                           X=X_train_fourier,
                           seasonal=False,
                           d=1,
                           stepwise=True,
                           suppress_warnings=True)

# 3. Prévision
forecast_fourier = model_fourier.predict(n_periods=24, X=X_test_fourier)

# 4. Visualisation
plt.figure(figsize=(14, 6))
plt.plot(np.arange(120, 144), test_sim, label="Réalité (Sinus Pur)", color='green', linestyle='--')
plt.plot(np.arange(120, 144), forecast_fourier, label="Approximation Trigonométrique (Fourier)", color='blue', linewidth=2)
plt.title("Approximation Déterministe via Séries de Fourier")
plt.legend()
plt.show()

print(model_fourier.summary())

#**Approfondissement : Mémoire Stochastique vs Approximation Harmonique**

Dans ce Lab, nous avons exploré deux philosophies radicalement différentes pour modéliser le même phénomène saisonnier.

**1. L'approche Stochastique (SARIMA) :** "La Mémoire"Le modèle SARIMA traite la saisonnalité comme un héritage. Il dit : "La valeur d'aujourd'hui est corrélée à celle d'il y a 12 mois".

Formellement : Il utilise l'opérateur de décalage saisonnier $L^{12}$ pour lier $y_t$ à $y_{t-12}$.

Limitation : Il ne connaît pas la notion de "courbe". Pour dessiner un sinus, il doit multiplier les coefficients ($P=2$) pour approximer la courbure de manière linéaire.

**2. L'approche Harmonique (Fourier**) : "L'Horloge"L'approche par séries de Fourier traite la saisonnalité comme un phénomène physique déterministe. On injecte directement des fonctions trigonométriques comme variables explicatives (Exogènes).

La Formule de l'Approximateur :Le modèle ne cherche plus à se souvenir de l'année passée, il ajuste une onde sur les données :$$y_t = \beta t + \sum_{k=1}^{K} \left[ \alpha_k \sin\left(\frac{2\pi kt}{12}\right) + \gamma_k \cos\left(\frac{2\pi kt}{12}\right) \right] + \eta_t$$Où :$\beta t$ : Capture la tendance linéaire.$\alpha, \gamma$ : Sont les coefficients de Fourier qui ajustent l'amplitude et la phase de l'onde.

$\eta_t$ : Est le résidu, modélisé par un simple ARIMA (Bruit).




Conclusion pour notre simulation : Comme nous avons généré la série avec un sinus parfait, l'approche de Fourier est ici supérieure et plus élégante. Elle "retrouve" la fonction génératrice originale là où le SARIMA devait l'imiter par la force des coefficients AR.

In [ ]:
# 1. Génération des régresseurs de Fourier pour le futur (36 mois)
# On se place juste après la fin des données de test (index 144)
n_forecast = 36
future_index = np.arange(144, 144 + n_forecast)

# Le featurizer renvoie parfois une simple liste, on force la conversion
X_future_fourier, _ = four_feat.transform(future_index)

# --- CORRECTION CRITIQUE : RESHAPE ---
# On s'assure que c'est bien une matrice (36 lignes, 2 colonnes)
X_future_fourier = np.array(X_future_fourier).reshape(n_forecast, -1)

# 2. Prédiction avec les exogènes correctement formatées
forecast_3y = model_fourier.predict(n_periods=n_forecast, X=X_future_fourier)

# 3. Visualisation Grand Angle
plt.figure(figsize=(15, 6))

# Historique (Simulé + Test)
plt.plot(np.arange(144), y_sim, label="Historique (Simulé)", color='black', alpha=0.3)

# La prédiction (3 ans de futur)
plt.plot(future_index, forecast_3y, label="Projection Harmonique (3 ans)", color='blue', linewidth=2, linestyle='--')

plt.title("Stabilité du Modèle de Fourier : Projection à 36 mois")
plt.xlabel("Temps (Mois)")
plt.ylabel("Valeur")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 🌐 Pont Trading Intelligent — saisonnalité financière : oui, mais autrement

Les niveaux de prix quotidiens ne présentent pas nécessairement une saisonnalité annuelle stable comparable à des ventes ou à une température. En revanche, les marchés peuvent présenter :

- effets de jour de semaine ;
- effets d'ouverture et de clôture ;
- saisonnalité intraday du volume ;
- volatilité intraday en forme de U ;
- effets de fin de mois ou d'échéances.

**Attention :** détecter une répétition historique ne prouve pas qu'elle constitue une anomalie rentable. Elle peut être instable, disparaître après coûts de transaction ou résulter du data snooping.

Dans CH03, nous modéliserons la variance conditionnelle par ARCH/GARCH. Avant cela, il faudra distinguer une composante périodique déterministe de volatilité d'un véritable clustering stochastique.


In [ ]:
# Exemple pédagogique : une volatilité intraday déterministe en forme de U
import numpy as np
import matplotlib.pyplot as plt

m = 390  # minutes approximatives d'une séance US régulière
u = np.linspace(-1, 1, m)
seasonal_sigma = 0.6 + 1.4 * (u**2)

plt.figure(figsize=(10,3.8))
plt.plot(seasonal_sigma)
plt.title("Exemple : profil saisonnier intraday de volatilité en forme de U")
plt.xlabel("Minute de séance")
plt.ylabel("Volatilité relative")
plt.grid(alpha=.2)
plt.show()

print("Cette courbe est déterministe : à heure identique, le profil se répète.")
print("Un GARCH cherche autre chose : une variance conditionnelle qui réagit aux chocs passés.")


#**1️⃣ Partie III : Le Challenge Air Passengers (Analyse Exploratoire)**

Nous allons maintenant appliquer notre savoir-faire sur le dataset **AirPassengers** (nombre mensuel de passagers aériens internationaux aux USA, 1949-1960).

C'est le classique car il présente un piège que nous n'avons pas encore vu : **l'hétéroscédasticité** (variance qui change).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pmdarima as pm
from pmdarima.datasets import load_airpassengers

# --- 1. CHARGEMENT ET CORRECTION DES DATES ---
# On charge la série brute
y_air = load_airpassengers(as_series=True)

# CORRECTION : On crée l'index temporel réel (Jan 1949 -> Déc 1960)
y_air.index = pd.date_range(start='1949-01-01', periods=len(y_air), freq='MS')

# --- 2. DIAGNOSTIC VISUEL (Mégaphone) ---
plt.figure(figsize=(14, 6))
plt.plot(y_air, label="Air Passengers (Données Brutes)", color='#8b5cf6')
plt.title("Le Problème : Observez l'amplitude des vagues qui grandit (1949-1960)")
plt.xlabel("Années")
plt.ylabel("Nombre de passagers")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

##**2️⃣ Le Diagnostic "Mégaphone"**
Le graphique montre clairement le problème : plus le temps passe, plus les vagues sont hautes. C'est l'hétéroscédasticité.

Maintenant, nous devons appliquer le "Remède" : **la Transformation Logarithmique.**
Cela va écraser les grandes valeurs plus fort que les petites, ce qui va "redresser" le mégaphone pour en faire un tube droit.

##3️⃣ **Le Remède : Transformation Logarithmique**

In [ ]:
# --- 3. LE REMÈDE : Transformation Logarithmique ---
# On applique le Log (Box-Cox avec lambda=0)
y_log = np.log(y_air)

# Visualisation Comparée
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Graphique 1 : Données Brutes (Mégaphone)
ax[0].plot(y_air, color='#8b5cf6')
ax[0].set_title("Avant : Variance Explosive (Multiplicatif)")
ax[0].set_xlabel("Années")
ax[0].grid(True, alpha=0.3)

# Graphique 2 : Données Log (Tube)
ax[1].plot(y_log, color='#ec4899')
ax[1].set_title("Après Log : Variance Stabilisée (Additif)")
ax[1].set_xlabel("Années")
ax[1].grid(True, alpha=0.3)

plt.show()

In [ ]:
# --- 4. PRÉPARATION (Train / Test) ---
# On garde les 2 dernières années (24 mois) pour tester la prévision plus tard
train_size = len(y_log) - 24
train_log, test_log = y_log[:train_size], y_log[train_size:]

print(f"Train : {len(train_log)} mois | Test : {len(test_log)} mois")

# --- 5. AUTO-ARIMA (Recherche du Modèle) ---
# Attention : On travaille sur 'train_log' !
print("⏳ Recherche du meilleur modèle SARIMA pour Air Passengers...")

model_air = pm.auto_arima(
    train_log,
    seasonal=True, m=12,  # Indispensable : cycle annuel
    d=1, D=1,             # On force la double différenciation (Tendance + Saison)
    start_p=0, start_q=0,
    max_p=2, max_q=2,
    max_P=2, max_Q=2,
    trace=True,           # Affiche les étapes
    error_action='ignore',
    suppress_warnings=True,
    stepwise=True
)

print("\n🏆 MEILLEUR MODÈLE :")
print(model_air.summary())

##L**e modèle a convergé vers ARIMA(0, 1, 1)(0, 1, 1)[12].**
ce modèle spécifique est ultra-célèbre dans la littérature scientifique. On l'appelle souvent le "Airline Model" de Box & Jenkins (1970). C'est la référence absolue pour cette série.

Il est très différent de notre simulation sinusoïdale (qui voulait des AR). Ici, le modèle préfère les Moyennes Mobiles (MA).

##**📝 Analyse des Résultats**

**L'Architecture "Airline" :(0, 1, 1)** : Une différenciation pour la tendance + un terme MA(1) pour lisser les chocs récents.

**(0, 1, 1)[12]** : Une différenciation saisonnière + un terme MA saisonnier pour lisser les chocs annuels.

**Pourquoi MA et pas AR ?** Contrairement à un sinus parfait (courbe rigide), les données économiques réelles sont souvent constituées de "chocs" qui s'estompent. Le modèle MA ("Moving Average") est meilleur pour modéliser cette absorption des chocs.

**Qualité Statistique :Ljung-Box (Prob=0.92)** : C'est un score excellent (proche de 1). Cela prouve qu'il ne reste absolument aucune information dans les résidus. Le modèle a tout "digéré".

**Heteroskedasticity (Prob=0.00)** : Le test détecte encore une variance instable malgré le Log. C'est normal sur cette série très volatile, mais le Log a fait l'essentiel du travail.

**🚀 L'Étape Finale** : La Prévision et le Retour au Réel

**Attention** ! Le modèle a travaillé sur des Logarithmes.Si le modèle prédit 6.1, cela ne veut pas dire "6 passagers", mais $e^{6.1} \approx 445$ passagers.Il faut impérativement utiliser np.exp() pour revenir à la réalité.

In [ ]:
# --- 6. PRÉVISION (Sur le Test Set) ---
# On prédit 24 mois (les 2 années cachées)
forecast_log, conf_int_log = model_air.predict(n_periods=len(test_log), return_conf_int=True)

# --- 7. INVERSE TRANSFORM (Log -> Réel) ---
# On revient à l'échelle "Nombre de Passagers"
forecast_real = np.exp(forecast_log)
conf_int_real = np.exp(conf_int_log)
train_real = np.exp(train_log)
test_real = np.exp(test_log)

# --- 8. VISUALISATION FINALE ---
plt.figure(figsize=(14, 7))

# Historique (Train) - Zoom sur la fin (1954-1958)
plt.plot(train_real.index[-60:], train_real[-60:], label="Historique (Train)", color='black')

# Réalité (Test) - (1959-1960)
plt.plot(test_real.index, test_real, label="Réalité (Cachée)", color='green', linestyle="--", linewidth=2)

# Prévision du Modèle
plt.plot(test_real.index, forecast_real, label="Prévision SARIMA (Airline Model)", color='red', linewidth=2)

# Intervalle de Confiance
plt.fill_between(
    test_real.index,
    conf_int_real[:, 0],
    conf_int_real[:, 1],
    color='pink', alpha=0.3, label="Intervalle 95%"
)

plt.title("Verdict Final : Prévision SARIMA sur Air Passengers (1959-1960)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

##**📝 Verdict Final : L'Analyse du "Gap"**
Ce graphique est riche en enseignements :

**La Victoire Structurelle** :
Le modèle a parfaitement anticipé le timing des pics (Juillet/Août) et des creux (Novembre). Il n'y a aucun décalage temporel.

**L'Effet du Logarithme** :
Regardez comment l'amplitude de la prévision (Rouge) continue de grandir entre 1959 et 1960. C'est grâce à la transformation np.exp() que nous avons faite à la fin. Sans cela, les vagues seraient restées plates.

**Pourquoi la Réalité dépasse-t-elle la Fiction ?**
Vous noterez que sur les pics d'été, la ligne verte dépasse légèrement la ligne rouge et frôle la sortie de la zone rose.

**Explicatio**n : Le modèle ARIMA est par nature "conservateur" (il moyenne le passé). Or, l'année 1960 a été une année de croissance exceptionnelle pour l'aviation, supérieure à la tendance moyenne des 10 années précédentes. Le modèle a été prudent, la réalité a été explosive.

**C'est une bonne nouvelle** : Le vrai trafic a été supérieur aux prévisions, tout en restant plausible (à la limite de l'intervalle de confiance).

## ✅ Checklist de compréhension — Lab 3

Vous devez pouvoir expliquer :

1. différence entre période, amplitude et phase ;
2. différence entre saisonnalité déterministe et stochastique ;
3. pourquoi une ACF saisonnière peut présenter des pics en \(s,2s,3s,\ldots\) ;
4. pourquoi \(d=1\) ne remplace pas \(D=1\) ;
5. sens de \((1-L^s)\) ;
6. signification de chaque terme dans \(SARIMA(p,d,q)	imes(P,D,Q)_s\) ;
7. pourquoi le logarithme aide lorsque l'amplitude saisonnière augmente avec le niveau ;
8. différence entre SARIMA et une représentation par termes de Fourier ;
9. pourquoi une saisonnalité financière observée n'est pas automatiquement une stratégie de trading.
